# HRS Silver CDM Master DML Functional Specification

---

## 1. Document Information

| Property           | Value                                              |
| ------------------ | -------------------------------------------------- |
| Document Name      | HRS Silver CDM Master DML Functional Specification |
| Version            | 1.0                                                |
| Author             | Perez                                              |
| AI Assistant       | ChatGPT                                            |
| Last Updated       | 2026-09-09                                         |
| Target Platform    | Databricks                                         |
| Compute            | Serverless                                         |
| Runtime            | client.5.12                                        |
| SQL Dialect        | Databricks SQL / Spark SQL                         |
| Storage Format     | Delta                                              |
| Specification Type | DML Only                                           |
| Target Data Layer  | Silver CDM                                         |

### 1.1 Purpose

This specification defines the functional and technical requirements for generating SQL Data Manipulation Language (DML) used to transform source RAND HRS data and load it into an existing Silver CDM target table.

The DML generated from this specification must conform to the target table structure defined by the corresponding Silver CDM DDL specification.

The specification is designed to be reusable across HRS Silver CDM subject-area tables.

---

## 2. DML Generation Objective

The generated DML must:

1. Read data from the specified source table.
2. Identify the appropriate respondent using the source natural identifier.
3. Identify the appropriate survey wave.
4. Transform the source data from the RAND HRS longitudinal structure into the target CDM grain.
5. Resolve system-generated parent-table surrogate keys.
6. Apply the specified source-to-target mappings.
7. Apply explicitly defined transformations.
8. Apply explicitly defined NULL and missing-value rules.
9. Populate required audit columns.
10. Validate the target business grain.
11. Load the transformed data into the existing target table.

The generated DML must not create or modify the physical definition of the target table.

---

# 3. Scope

## 3.1 Included

The generated DML may contain:

* `SELECT`
* `INSERT INTO`
* Common Table Expressions (`WITH`)
* `JOIN`
* `LEFT JOIN`
* `INNER JOIN`
* `CROSS JOIN`
* `UNION ALL`
* `CASE`
* `CAST`
* Explicit data-type conversion
* Source filtering
* Wave transformation
* Unpivoting
* Natural-identifier resolution
* Parent-key resolution
* Business-rule transformations
* NULL handling
* Missing-value handling
* Audit-column population
* Duplicate detection and prevention when explicitly specified

## 3.2 Excluded

The generated DML must not contain:

* `CREATE TABLE`
* `DROP TABLE`
* `ALTER TABLE`
* `TRUNCATE TABLE`
* `CREATE VIEW`
* `CREATE INDEX`
* `PARTITION BY`
* `ZORDER`
* `OPTIMIZE`

The generated DML must also not contain:

* `UPDATE`
* `DELETE`
* `MERGE`

unless explicitly authorized by a subject-area specification.

The default DML load pattern is **Insert Only**.

---

# 4. Source Data Parameters

The source parameters identify the physical source data object.

| Parameter                  | Value                                        |
| -------------------------- | -------------------------------------------- |
| `SOURCE_TABLE_NAME`        | `dev_catalog.brz_raw_hrs.randhrs1992_2022v1` |
| `SOURCE_TABLE_DESCRIPTION` | RAND HRS Longitudinal Dataset                |
| `SOURCE_CATALOG_NAME`      | `dev_catalog`                                |
| `SOURCE_SCHEMA_NAME`       | `brz_raw_hrs`                                |

### 4.1 Source Parameter Usage

The parameters defined above must be used when referencing the source data.

The generated DML must not hard-code alternate catalog, schema, or table names.

If the source table changes, the parameter values must be updated before DML generation.

---

# 5. Target Data Parameters

The target parameters identify the existing Silver CDM table that will receive the transformed data.

| Parameter                  | Value                                         |
| -------------------------- | --------------------------------------------- |
| `TARGET_TABLE_NAME`        | `dev_catalog.slv_cdm_hrs.<TARGET_TABLE_NAME>` |
| `TARGET_CATALOG_NAME`      | `dev_catalog`                                 |
| `TARGET_SCHEMA_NAME`       | `slv_cdm_hrs`                                 |
| `TARGET_TABLE_DESCRIPTION` | `<TARGET_TABLE_DESCRIPTION>`                  |
| `LOAD_PATTERN`             | Insert Only                                   |

### 5.1 Target Parameter Usage

The parameters defined above must be used throughout the DML generation process.

The generated DML must load only the specified target table.

The DML generator must not create the target table.

The target table is assumed to have already been created successfully by the corresponding DDL script.

---

# 6. Target Business Grain

The target business grain defines what one target row represents.

### 6.1 Default HRS Silver CDM Grain

Unless explicitly overridden by a subject-area specification:

> One target row represents one respondent for one survey wave.

The logical business key is:

```text
respondent_id + wave_id
```

### 6.2 Natural Identifier Grain

The source natural identifiers used to locate the parent surrogate keys are:

| Natural Identifier | Purpose                    |
| ------------------ | -------------------------- |
| `HHIDPN`           | Identifies the respondent  |
| `wave_number`      | Identifies the survey wave |

The relationship is:

```text
HHIDPN
   ↓
hub_respondent
   ↓
respondent_id
```

and:

```text
wave_number
   ↓
dim_wave
   ↓
wave_id
```

The generated DML must not generate or calculate `respondent_id` or `wave_id`.

---

# 7. Parent Table Dependencies

The following parent tables are required unless overridden by the subject-area specification.

| Parent Table     | Purpose                                                   |
| ---------------- | --------------------------------------------------------- |
| `hub_respondent` | Resolves respondent natural identifier to `respondent_id` |
| `dim_wave`       | Resolves wave natural identifier to `wave_id`             |

### 7.1 Parent Table Locations

| Parameter             | Value         |
| --------------------- | ------------- |
| `PARENT_CATALOG_NAME` | `dev_catalog` |
| `PARENT_SCHEMA_NAME`  | `slv_cdm_hrs` |

### 7.2 Parent Table Primary Keys

| Parent Table     | Primary Key     |
| ---------------- | --------------- |
| `hub_respondent` | `respondent_id` |
| `dim_wave`       | `wave_id`       |

---

# 8. Parent-Key Resolution Rules

## 8.1 Respondent Key Resolution

The source `HHIDPN` must be used to locate the corresponding respondent in `hub_respondent`.

Conceptually:

```text
SOURCE.HHIDPN
      ↓
hub_respondent.HHIDPN
      ↓
hub_respondent.respondent_id
      ↓
TARGET.respondent_id
```

The generated DML must use the parent table's existing `respondent_id`.

The DML must not generate a new `respondent_id`.

## 8.2 Wave Key Resolution

The source `wave_number` must be used to locate the corresponding wave in `dim_wave`.

Conceptually:

```text
SOURCE.wave_number
      ↓
dim_wave.wave_number
      ↓
dim_wave.wave_id
      ↓
TARGET.wave_id
```

The generated DML must use the parent table's existing `wave_id`.

The DML must not generate a new `wave_id`.

## 8.3 Unresolved Parent Keys

The subject-area specification must explicitly define how unresolved parent keys are handled.

The default rule is:

> A source record that cannot be resolved to a valid parent key must not be silently loaded into the target table.

The DML generator must not invent a surrogate key or substitute a default surrogate key.

---

# 9. Source Data Grain and Structure

The RAND HRS source data is generally structured as a wide longitudinal dataset.

Wave-specific variables commonly follow the pattern:

```text
R1<VARIABLE>
R2<VARIABLE>
R3<VARIABLE>
...
R16<VARIABLE>
```

The Silver CDM target structure is longitudinal at the respondent-wave level.

Therefore, the DML may need to transform the source from:

```text
One source respondent row
        ↓
Multiple wave observations
```

into:

```text
respondent + wave 1
respondent + wave 2
respondent + wave 3
...
respondent + wave N
```

The DML specification must explicitly identify which source variables are wave-specific and which are wave-invariant.

The generator must not infer this distinction when the specification does not provide sufficient information.

---

# 10. Wave Transformation Rules

## 10.1 Wave-Specific Variables

For wave-specific source variables, the source variable must be associated with the corresponding survey wave.

Example:

| Source Variable | Wave |
| --------------- | ---: |
| `R1AGEY_E`      |  `1` |
| `R2AGEY_E`      |  `2` |
| `R3AGEY_E`      |  `3` |
| `R4AGEY_E`      |  `4` |

The generated DML must transform these variables into the target attribute:

```text
agey_e
```

while retaining the corresponding:

```text
wave_number
```

## 10.2 Wave Number Data Type

`wave_number` is a `STRING`.

The DML must therefore treat the wave identifier as a string when joining to `dim_wave`.

The DML must not assume that `wave_number` is an integer.

## 10.3 Wave Enumeration

The subject-area specification must identify the supported survey waves.

Example:

```text
Wave 1 through Wave 16
```

The generator must generate only the waves specified by the subject-area mapping.

---

# 11. Wave-Invariant Variables

Some RAND HRS variables occur once per respondent rather than once per survey wave.

Examples may include:

```text
RARACEM
RAHISPAN
RAEDYRS
RARELIG
RAVETRN
```

The subject-area specification must identify variables that are wave-invariant.

For each wave-invariant variable, the specification must define how the value is associated with respondent-wave observations.

The DML generator must not independently assume that a wave-invariant variable should be replicated across all waves.

---

# 12. Source-to-Target Mapping Matrix

The source-to-target mapping matrix is mandatory for DML generation.

Each target business column must have an explicitly defined mapping.

| Wave     | Source Variable     | Variable Label | RAND Type     | Target Column     | Databricks Type | Transformation     | Nullable | Missing-Value Rule |
| -------- | ------------------- | -------------- | ------------- | ----------------- | --------------- | ------------------ | -------- | ------------------ |
| `<wave>` | `<source_variable>` | `<label>`      | `<RAND type>` | `<target_column>` | `<type>`        | `<transformation>` | Yes/No   | `<rule>`           |

### 12.1 Mapping Requirements

Each mapping must identify:

1. Source variable
2. Target column
3. Source data type or RAND type
4. Target Databricks data type
5. Transformation rule
6. NULL behavior
7. Missing-value behavior
8. Wave applicability

### 12.2 Missing Mapping Rule

The DML generator must not invent a source-to-target mapping.

If a required target column does not have a defined source mapping, the generator must flag the missing mapping rather than infer one.

---

# 13. Transformation Rules

All transformations must be explicitly defined.

Permitted transformation examples include:

```text
DIRECT
CAST
CASE
NULLIF
COALESCE
STRING CONVERSION
NUMERIC CONVERSION
DATE CONVERSION
WAVE EXTRACTION
UNPIVOT
```

### 13.1 Direct Mapping

A direct mapping indicates that the source value is transferred without business-rule transformation.

Example:

```text
R1AGEY_E → agey_e
```

### 13.2 Data-Type Conversion

When source and target data types differ, the specification must define the required conversion.

Example:

```text
Source: DOUBLE
Target: DECIMAL(10,2)
Transformation: CAST
```

The generated SQL must use an explicit cast when required.

### 13.3 Business Transformations

Business transformations must be specified explicitly.

The generator must not infer business meaning from a variable name alone.

---

# 14. NULL Handling

The default rule is:

> Source SQL NULL values remain NULL unless the subject-area specification explicitly defines another treatment.

The DML generator must not automatically convert NULL values to:

```text
0
-1
999
'Unknown'
'Not Available'
```

unless explicitly instructed.

---

# 15. RAND HRS Missing-Value Rules

RAND HRS variables may contain values representing different forms of missing or non-response data.

The subject-area specification must explicitly identify how these values are handled.

Possible categories include:

| Source Condition   | Target Treatment |
| ------------------ | ---------------- |
| Valid value        | Preserve         |
| SQL NULL           | NULL             |
| Missing-value code | `<Defined Rule>` |
| Don't Know         | `<Defined Rule>` |
| Refused            | `<Defined Rule>` |
| Not Applicable     | `<Defined Rule>` |
| Not Asked          | `<Defined Rule>` |
| Other special code | `<Defined Rule>` |

The DML generator must not assume that a numeric value represents a missing condition without an explicit rule.

---

# 16. Audit Column Rules

Unless overridden by the subject-area specification, the following rules apply:

| Target Column | DML Rule         |
| ------------- | ---------------- |
| `create_date` | `CURRENT_DATE()` |
| `update_date` | `CURRENT_DATE()` |
| `active`      | `TRUE`           |

The audit columns must be populated during the target load.

---

# 17. Identity Column Rule

The target identity column must never be populated by the DML.

For example:

```text
fact_demographics_id
```

must be omitted from the target `INSERT` column list.

Databricks generates the identity value automatically.

The DML must not:

* calculate the identity value
* select an identity value from the source
* insert a hard-coded identity value
* use `MAX(identity_column) + 1`

---

# 18. Load Strategy

The default target load pattern is:

```text
INSERT ONLY
```

The generated DML must use:

```sql
INSERT INTO
```

to load the target table.

The DML must not update or delete existing target records.

The DML must not use `MERGE` unless explicitly authorized by a future subject-area specification.

---

# 19. Insert Column Requirements

The generated `INSERT` statement must explicitly identify target columns.

The generator must not rely on the physical column order of the target table.

Example structure:

```sql
INSERT INTO <TARGET_TABLE_NAME>
(
    respondent_id,
    wave_id,
    create_date,
    update_date,
    active,
    HHIDPN,
    wave_number,
    <business_columns>
)
SELECT
    ...
```

The identity column must be excluded from the insert list.

---

# 20. Duplicate Handling

The target business grain is:

```text
respondent_id + wave_id
```

Therefore, the DML must prevent multiple source records from producing the same target business observation.

The subject-area specification must define the appropriate duplicate-handling rule.

Possible rules include:

* Reject duplicates
* Deduplicate using a defined source priority
* Select a defined source record
* Aggregate records
* Exclude duplicate observations

The DML generator must not silently select an arbitrary record.

---

# 21. Source Filtering Rules

Source filtering must be explicitly defined.

The subject-area specification may specify filters based on:

* Respondent eligibility
* Survey wave
* Respondent type
* Population
* Source status
* Valid source identifiers

If no filter is specified, the generator must not introduce additional business filters.

---

# 22. DML Processing Sequence

The generated DML should follow this logical processing sequence:

```text
1. Read source data
        ↓
2. Identify source respondents
        ↓
3. Identify applicable survey waves
        ↓
4. Transform/unpivot wave-specific variables
        ↓
5. Apply source-to-target mappings
        ↓
6. Apply data-type conversions
        ↓
7. Resolve respondent_id
        ↓
8. Resolve wave_id
        ↓
9. Apply wave-invariant variables
        ↓
10. Apply NULL and missing-value rules
        ↓
11. Populate audit columns
        ↓
12. Validate business grain
        ↓
13. Insert into target table
```

The generated SQL may implement these steps using CTEs or equivalent SQL structures.

---

# 23. Recommended CTE Structure

When practical, generated DML should use logical CTE stages.

A recommended pattern is:

```text
source_data
    ↓
respondent_resolution
    ↓
wave_transformation
    ↓
attribute_mapping
    ↓
parent_key_resolution
    ↓
final_transformation
    ↓
validated_rows
    ↓
INSERT
```

CTE names should be descriptive and use consistent `snake_case` naming.

---

# 24. Referential Integrity Validation

Before loading the target table, the DML process must validate parent-key resolution.

### Respondent Validation

Every target `respondent_id` must correspond to a valid record in:

```text
dev_catalog.slv_cdm_hrs.hub_respondent
```

### Wave Validation

Every target `wave_id` must correspond to a valid record in:

```text
dev_catalog.slv_cdm_hrs.dim_wave
```

Unresolved keys must be identified rather than silently substituted.

---

# 25. Business-Grain Validation

The generated DML must validate that:

```text
respondent_id + wave_id
```

uniquely identifies a target observation.

The validation must identify duplicate combinations before they are loaded.

Example validation concept:

```sql
SELECT
    respondent_id,
    wave_id,
    COUNT(*) AS record_count
FROM <transformed_data>
GROUP BY
    respondent_id,
    wave_id
HAVING COUNT(*) > 1;
```

The final implementation may use a different technique, but the business rule must be preserved.

---

# 26. DML Validation Requirements

| Validation                           | Required |
| ------------------------------------ | -------- |
| Source table exists                  | ✓        |
| Target table exists                  | ✓        |
| Parent respondent table exists       | ✓        |
| Parent wave table exists             | ✓        |
| Source HHIDPN resolves               | ✓        |
| Source wave_number resolves          | ✓        |
| Target respondent_id populated       | ✓        |
| Target wave_id populated             | ✓        |
| Identity column excluded from INSERT | ✓        |
| Required audit columns populated     | ✓        |
| Target business columns mapped       | ✓        |
| Data types correctly converted       | ✓        |
| NULL rules applied                   | ✓        |
| Missing-value rules applied          | ✓        |
| No duplicate respondent_id + wave_id | ✓        |
| Expected wave coverage               | ✓        |
| Expected respondent coverage         | ✓        |

---

# 27. Reconciliation Requirements

The generated DML should support reconciliation between source and target data.

At minimum, the validation process should be capable of comparing:

### 27.1 Respondent Counts

```text
Source distinct HHIDPN
        vs.
Target distinct respondent_id
```

### 27.2 Respondent-Wave Counts

```text
Source respondent-wave observations
        vs.
Target respondent-wave observations
```

### 27.3 Wave Counts

```text
Source observations by wave
        vs.
Target observations by wave
```

Subject-area specifications may define expected counts or acceptable differences.

---

# 28. Error Handling Requirements

The generated DML must not silently discard records because of:

* Missing respondent key
* Missing wave key
* Invalid transformation
* Duplicate business key
* Invalid data conversion

Where practical, validation SQL should identify rejected or unresolved records so they can be investigated.

The DML generator must not invent replacement values for unresolved identifiers.

---

# 29. SQL Formatting Requirements

| Requirement                        | Required |
| ---------------------------------- | -------- |
| `INSERT INTO`                      | ✓        |
| Explicit target columns            | ✓        |
| `SELECT`                           | ✓        |
| CTEs when beneficial               | ✓        |
| Explicit JOIN conditions           | ✓        |
| Explicit CAST when required        | ✓        |
| Uppercase SQL keywords             | ✓        |
| Consistent indentation             | ✓        |
| Descriptive CTE names              | ✓        |
| Fully qualified table names        | ✓        |
| No implicit target column ordering | ✓        |

---

# 30. Unsupported DML Objects and Operations

The generated DML must not contain the following unless explicitly authorized:

```text
CREATE TABLE
DROP TABLE
ALTER TABLE
TRUNCATE TABLE
CREATE VIEW
CREATE INDEX
UPDATE
DELETE
MERGE
PARTITION BY
ZORDER
OPTIMIZE
```

The DML specification is responsible only for loading data into an existing target table.

---

# 31. Parameter Substitution Rules

Parameters defined in the specification may be referenced using the following format:

```text
<TARGET_TABLE_NAME>
<SOURCE_TABLE_NAME>
<PARENT_CATALOG_NAME>
<PARENT_SCHEMA_NAME>
<RESPONDENT_PARENT_TABLE>
<WAVE_PARENT_TABLE>
```

The SQL generator must substitute parameter values before producing the final SQL.

The generated SQL must contain resolved object names and must not contain unresolved template placeholders.

---

# 32. Subject-Area DML Requirements

Each Silver CDM subject-area DML specification must provide the following information:

1. Target table name
2. Target table description
3. Target business grain
4. Source table
5. Parent tables
6. Natural identifiers
7. Parent-key resolution rules
8. Applicable survey waves
9. Wave-specific variables
10. Wave-invariant variables
11. Source-to-target mappings
12. Data-type transformations
13. NULL rules
14. RAND missing-value rules
15. Source filters
16. Duplicate handling
17. Audit-column rules
18. Reconciliation requirements

The generator must not infer missing business rules.

---

# 33. DML Generation Decision Rules

The SQL generator must follow these rules:

### Rule 1 — Do Not Invent Mappings

If a source-to-target mapping is missing, flag it.

### Rule 2 — Do Not Invent Business Rules

If the treatment of a source value is ambiguous, do not infer its business meaning.

### Rule 3 — Do Not Generate Surrogate Keys

`respondent_id` and `wave_id` must always be resolved from their parent tables.

### Rule 4 — Do Not Populate Identity Columns

Identity columns must be omitted from the target insert list.

### Rule 5 — Preserve NULL Unless Directed Otherwise

Do not automatically replace NULL with a default value.

### Rule 6 — Treat Wave Number as STRING

Do not implicitly cast `wave_number` to an integer for parent-table resolution.

### Rule 7 — Respect the Target Grain

The generated DML must produce no more than one target observation for each:

```text
respondent_id + wave_id
```

unless explicitly authorized.

### Rule 8 — Do Not Silently Drop Data

Records failing required parent-key or transformation rules must be identifiable.

### Rule 9 — DML Must Match DDL

Every target column populated by DML must exist in the corresponding DDL.

### Rule 10 — DML Must Not Modify Table Structure

The generated output must contain only the operations required to populate the existing target table.

---

# 34. Generated SQL Structure

Unless the subject-area specification requires another approach, the generated DML should follow this general structure:

```sql
WITH source_data AS
(
    SELECT
        ...
    FROM
        <SOURCE_TABLE_NAME>
),

respondent_resolution AS
(
    SELECT
        ...
    FROM
        source_data s
        LEFT JOIN <RESPONDENT_PARENT_TABLE> r
            ON s.HHIDPN = r.HHIDPN
),

wave_transformation AS
(
    SELECT
        ...
    FROM
        respondent_resolution
),

wave_resolution AS
(
    SELECT
        ...
    FROM
        wave_transformation w
        LEFT JOIN <WAVE_PARENT_TABLE> d
            ON w.wave_number = d.wave_number
),

final_data AS
(
    SELECT
        ...
    FROM
        wave_resolution
)

INSERT INTO <TARGET_TABLE_NAME>
(
    respondent_id,
    wave_id,
    create_date,
    update_date,
    active,
    HHIDPN,
    wave_number,
    ...
)
SELECT
    respondent_id,
    wave_id,
    CURRENT_DATE(),
    CURRENT_DATE(),
    TRUE,
    HHIDPN,
    wave_number,
    ...
FROM final_data;
```

This is a **structural template only**. The actual generated DML must be based on the subject-area mapping and transformation requirements.

---

# 35. DML Deliverables

| Item         | Value                                   |
| ------------ | --------------------------------------- |
| SQL File     | `/sql/dml/load_<TARGET_TABLE_NAME>.sql` |
| Output       | SQL Only                                |
| SQL Type     | DML                                     |
| Load Pattern | Insert Only                             |
| Target       | Existing Silver CDM Delta Table         |

---

# 36. Final DML Generation Requirement

The generated SQL must be production-ready Databricks SQL compatible with the specified environment:

```text
Platform: Databricks
Compute: Serverless
Runtime: client.5.12
Storage: Delta
Target Layer: Silver CDM
```

The generated DML must implement only the transformations and business rules explicitly defined in the applicable subject-area DML specification.

Where the specification does not provide sufficient information to determine the correct transformation, mapping, filtering, missing-value treatment, or business rule, the generator must identify the missing requirement rather than make an unsupported assumption.

**Return only SQL when generating the final DML script.**

```

This gives you a clean two-document architecture:

**Master DDL Specification**
→ defines **what the table is**

**Master DML Specification**
→ defines **how the table is populated**

And then, for example:

`HRS Demographics DML Specification v1.0`  
`HRS Health DML Specification v1.0`  
`HRS Wealth DML Specification v1.0`

would inherit the master rules and only supply the **subject-specific mappings, waves, transformations, and business rules**.
```
